# Aula 3, GloVe

Notebook executável que acompanha a aula [03-glove.md](../../lessons/modulo-04-word-embeddings/03-glove.md).

## O que vamos fazer aqui

O GloVe chega aos embeddings por contagem, não por previsão. Vamos montar a matriz de
coocorrência do mesmo corpus de gato e cachorro e fatorá-la com SVD, obtendo vetores
densos de palavra. Como gato e cachorro têm contextos quase idênticos, esperamos vê-los
praticamente colados. Só numpy.

## Matriz de coocorrência

A matriz de coocorrência conta, para cada par de palavras, quantas vezes elas aparecem
perto uma da outra dentro de uma janela. Ela resume, em um só lugar, todo o
comportamento de vizinhança do corpus.

In [ ]:
import numpy as np
import re

frases = [
    "o gato come peixe", "o cachorro come carne",
    "o gato bebe leite", "o cachorro bebe agua",
    "o gato dorme no sofa", "o cachorro dorme no tapete",
    "a crianca gosta do gato", "a crianca gosta do cachorro",
]


def tokenizar(texto):
    return re.findall(r"\w+", texto.lower())


tokens_frases = [tokenizar(f) for f in frases]
vocab = sorted({w for fr in tokens_frases for w in fr})
vi = {w: i for i, w in enumerate(vocab)}
V = len(vocab)

CO = np.zeros((V, V))
for fr in tokens_frases:
    ids = [vi[w] for w in fr]
    for i, centro in enumerate(ids):
        for j in range(max(0, i - 2), min(len(ids), i + 3)):
            if j != i:
                CO[centro, ids[j]] += 1

print("matriz de coocorrência:", CO.shape)

A matriz é quadrada, com uma linha e uma coluna por palavra do vocabulário.
Palavras de sentido próximo tendem a ter linhas parecidas, e é isso que a fatoração vai
explorar.

## Fatoração por SVD e similaridade

Suavizamos as contagens com o logaritmo, para conter o peso das palavras muito
frequentes, e aplicamos o SVD. Ficamos com as primeiras componentes como vetores
densos de palavra, no espírito do GloVe.

In [ ]:
M = np.log1p(CO)
U, S, Vt = np.linalg.svd(M)
k = 5
E = U[:, :k] * S[:k]   # vetores densos de palavra


def similaridade(a, b):
    va, vb = E[vi[a]], E[vi[b]]
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb)))


print("gato ~ cachorro:", round(similaridade("gato", "cachorro"), 3))
print("come ~ bebe    :", round(similaridade("come", "bebe"), 3))
print("gato ~ peixe   :", round(similaridade("gato", "peixe"), 3))

Gato e cachorro aparecem com similaridade altíssima, perto de 1, assim como
come e bebe, enquanto gato e peixe ficam bem mais baixos. O resultado tão extremo vem
do corpus minúsculo e simétrico, em que gato e cachorro são quase intercambiáveis. Em
um corpus grande, as similaridades seriam mais matizadas, mas o mecanismo é o mesmo, e
chega ao destino do Word2Vec por um caminho de contagem. Para o projeto, compare estas
similaridades com as do skip-gram da Aula 1.